# Exploração da base de avaliações da Olist

Antes de preparar os dados e construir a busca, preciso conhecer a base com que vou trabalhar: quantas avaliações têm texto, como são esses textos e quais informações de apoio, como nota, categoria e entrega, estão disponíveis para cada uma. Os números levantados aqui embasam as decisões de preparação do projeto e serão citados no README.

Começo com uma visão geral da base (seções 1 a 8), olhando a quantidade de comentários, a distribuição das notas, o tamanho e a repetição dos textos, os identificadores, as categorias e as datas de entrega. Em seguida, cruzo a nota com o atraso na entrega, com as categorias e com o período (seções 9 a 11). Depois analiso o conteúdo dos comentários: palavras e temas mais frequentes, tamanho por nota, exemplos reais, menções a aplicativo e caracteres especiais (seções 12 a 17). Por fim, aplico as decisões de seleção para chegar à base final, verificando quantos documentos sobram, quanto dos metadados fica disponível para os filtros, quanto custa gerar os embeddings e como são os títulos (seções 18 a 21). A conclusão, no fim do notebook, resume o que encontrei e as decisões que esses resultados motivaram.

Para rodar, baixe a base com `python -m scripts.download_data` a partir da raiz do projeto e selecione o ambiente `.venv` como kernel.

In [1]:
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")

reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")

reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53


## 1. Avaliações com e sem comentário

A tabela de avaliações tem uma linha por avaliação, e o texto escrito pelo cliente pode estar em duas colunas: `review_comment_title`, o título, e `review_comment_message`, a mensagem. Quando o cliente não escreveu nada, o valor fica vazio (`NaN`).

In [2]:
has_message = reviews["review_comment_message"].notna()
has_title = reviews["review_comment_title"].notna()

pd.Series({
    "Total de avaliações": len(reviews),
    "Com mensagem": has_message.sum(),
    "Com título": has_title.sum(),
    "Com título e mensagem": (has_title & has_message).sum(),
    "Só com título": (has_title & ~has_message).sum(),
    "Sem nenhum texto": (~has_title & ~has_message).sum(),
})

Total de avaliações      99224
Com mensagem             40977
Com título               11568
Com título e mensagem     9839
Só com título             1729
Sem nenhum texto         56518
dtype: int64

Decidi usar na base de conhecimento apenas as avaliações com mensagem. As que têm só título ficam de fora, porque um título sozinho raramente explica o que aconteceu com a compra.

## 2. Distribuição das notas

Comparo as notas de todas as avaliações com as notas das avaliações que têm comentário, para saber se quem escreve tem um perfil diferente de quem só dá a nota.

In [3]:
scores = pd.DataFrame({
    "Todas": reviews["review_score"].value_counts(normalize=True).sort_index(),
    "Com mensagem": reviews.loc[has_message, "review_score"].value_counts(normalize=True).sort_index(),
})
(scores * 100).round(1)

,Todas,Com mensagem
review_score,,
1,11.5,21.3
2,3.2,5.2
3,8.2,8.7
4,19.3,14.6
5,57.8,50.2


## 3. Tamanho dos comentários

Antes de medir, aplico a mesma limpeza leve que pretendo usar na preparação: quebras de linha e espaços repetidos viram um espaço só, e os espaços das pontas são removidos.

In [4]:
messages = (
    reviews.loc[has_message, "review_comment_message"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
lengths = messages.str.len()

lengths.describe().round(1)

count    40977.0
mean        68.2
std         53.7
min          0.0
25%         28.0
50%         53.0
75%         94.0
max        208.0
Name: review_comment_message, dtype: float64

In [5]:
pd.Series({f"Até {n} caracteres": (lengths <= n).sum() for n in [0, 3, 5, 10, 20]})

Até 0 caracteres       27
Até 3 caracteres      836
Até 5 caracteres     1497
Até 10 caracteres    3671
Até 20 caracteres    7299
dtype: int64

In [6]:
# Comentários que não têm nenhuma letra, como "." ou "..."
no_letters = ~messages.str.contains(r"[A-Za-zÀ-ÿ]", regex=True)
print(f"Comentários sem nenhuma letra: {no_letters.sum()}")
messages[no_letters].value_counts().head(10)

Comentários sem nenhuma letra: 186


review_comment_message
.        52
         27
10       19
...      11
..        8
100%      5
.....     3
👍🏻        3
?         3
-         3
Name: count, dtype: int64

## 4. Comentários repetidos

Muitos clientes escrevem a mesma coisa, como "Muito bom" ou "Recomendo". Aqui meço quantos textos se repetem, porque isso influencia como a busca vai se comportar e se vale a pena tratar os repetidos de forma diferente.

In [7]:
print(f"Comentários: {len(messages)}")
print(f"Textos distintos: {messages.nunique()}")
messages.value_counts().head(20)

Comentários: 40977
Textos distintos: 35616


review_comment_message
Muito bom               305
Bom                     215
Ótimo                   158
Recomendo               153
muito bom               144
bom                     111
otimo                   108
Excelente               108
Otimo                   107
Ok                       78
recomendo                71
Ótimo produto            67
ok                       58
Boa                      55
.                        52
Não recebi o produto     51
Tudo ok                  51
Gostei                   46
Muito bom.               43
excelente                40
Name: count, dtype: int64

Na lista acima aparecem variações do mesmo texto que só mudam nas maiúsculas ou nos acentos, como "Muito bom" e "muito bom". Na célula seguinte, conto quantos textos distintos sobram quando essas variações são consideradas iguais.

In [8]:
import unicodedata

def normalize_for_comparison(text: str) -> str:
    # Minúsculas e sem acentos, usado só para comparar textos
    text = unicodedata.normalize("NFKD", text.lower())
    return "".join(c for c in text if not unicodedata.combining(c))

print(f"Textos distintos (comparação exata): {messages.nunique()}")
print(f"Textos distintos (ignorando maiúsculas e acentos): {messages.map(normalize_for_comparison).nunique()}")

Textos distintos (comparação exata): 35616
Textos distintos (ignorando maiúsculas e acentos): 34864


## 5. Identificadores das avaliações

Cada documento da base precisa de um identificador único. Verifico aqui se o `review_id` e o `order_id` se repetem na tabela.

In [9]:
pd.Series({
    "review_id repetidos": reviews["review_id"].duplicated().sum(),
    "order_id repetidos": reviews["order_id"].duplicated().sum(),
})

review_id repetidos    814
order_id repetidos     551
dtype: int64

In [10]:
# Exemplo de review_id que aparece em mais de uma linha
repeated_id = reviews.loc[reviews["review_id"].duplicated(), "review_id"].iloc[0]
reviews[reviews["review_id"] == repeated_id]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
1009,3242cc306a9218d0377831e175d62fbf,711246846695fa895c1776f0ddc30b20,5,NaN,Ótimo,2017-07-21 00:00:00,2017-07-22 01:06:37
3317,3242cc306a9218d0377831e175d62fbf,9c5bfba7de6a4abbb6ba0baab78d1622,5,NaN,Ótimo,2017-07-21 00:00:00,2017-07-22 01:06:37


## 6. Pedidos com mais de uma categoria

A avaliação é feita para o pedido, mas a categoria pertence ao produto. Quando um pedido tem produtos de categorias diferentes, não dá para saber de qual deles o cliente está falando, então preciso saber quantos pedidos estão nessa situação.

In [11]:
items_with_category = items.merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left"
)
categories_per_order = items_with_category.groupby("order_id")["product_category_name"].nunique()

orders_with_message = reviews.loc[has_message, "order_id"].unique()
categories = categories_per_order.reindex(orders_with_message)

pd.Series({
    "Pedidos com mensagem": len(orders_with_message),
    "Com uma categoria": (categories == 1).sum(),
    "Com mais de uma categoria": (categories > 1).sum(),
    "Sem itens ou sem categoria": ((categories == 0) | categories.isna()).sum(),
})

Pedidos com mensagem          40836
Com uma categoria             39273
Com mais de uma categoria       452
Sem itens ou sem categoria     1111
dtype: int64

## 7. Datas de entrega

Considero que um pedido atrasou quando a data real de entrega (`order_delivered_customer_date`) é posterior à data estimada (`order_estimated_delivery_date`). Quando a data real não existe, o atraso fica como "desconhecido".

In [12]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [13]:
delivered = pd.to_datetime(orders["order_delivered_customer_date"])
estimated = pd.to_datetime(orders["order_estimated_delivery_date"])

pd.Series({
    "Pedidos": len(orders),
    "Sem data de entrega": delivered.isna().sum(),
    "Entregues com atraso": (delivered > estimated).sum(),
    "Entregues no prazo": (delivered <= estimated).sum(),
})

Pedidos                 99441
Sem data de entrega      2965
Entregues com atraso     7827
Entregues no prazo      88649
dtype: int64

## 8. Período das avaliações

Por último nesta visão geral, verifico o intervalo de datas coberto pela base e como as avaliações se distribuem por trimestre.

In [14]:
dates = pd.to_datetime(reviews["review_creation_date"])
print(f"De {dates.min():%d/%m/%Y} a {dates.max():%d/%m/%Y}")
dates.dt.to_period("Q").value_counts().sort_index()

De 02/10/2016 a 31/08/2018


review_creation_date
2016Q4      325
2017Q1     4133
2017Q2     9205
2017Q3    12205
2017Q4    17192
2018Q1    20083
2018Q2    21460
2018Q3    14621
Freq: Q-DEC, Name: count, dtype: int64

## 9. Nota das entregas atrasadas e das entregues no prazo

Ligo cada avaliação ao seu pedido para saber se a entrega atrasou e comparo as notas nas duas situações. Se o atraso pesar muito na satisfação, isso justifica oferecer um filtro de entrega atrasada.

In [15]:
delivery = orders[["order_id"]].copy()
delivery["delivery_status"] = "desconhecido"
delivery.loc[delivered <= estimated, "delivery_status"] = "no prazo"
delivery.loc[delivered > estimated, "delivery_status"] = "atrasado"

reviews_delivery = reviews.merge(delivery, on="order_id", how="left")
reviews_delivery["delivery_status"] = reviews_delivery["delivery_status"].fillna("desconhecido")

reviews_delivery.groupby("delivery_status")["review_score"].agg(["count", "mean"]).round(2)

,count,mean
delivery_status,,
atrasado,7701,2.57
desconhecido,2865,1.76
no prazo,88658,4.29


In [16]:
# Distribuição das notas em cada situação de entrega, em porcentagem
(pd.crosstab(reviews_delivery["delivery_status"], reviews_delivery["review_score"], normalize="index") * 100).round(1)

review_score,1,2,3,4,5
delivery_status,,,,,
atrasado,46.1,7.9,11.4,12.4,22.2
desconhecido,70.3,7.3,7.6,5.4,9.4
no prazo,6.6,2.6,8.0,20.3,62.4


## 10. Categorias com mais comentários

Aqui considero só os pedidos com uma única categoria, já que os pedidos mistos vão ficar fora do filtro de categoria. Uma categoria com poucos comentários rende buscas pobres quando filtrada, então quero saber quantas têm volume suficiente.

In [17]:
single_category = (
    items_with_category.groupby("order_id")["product_category_name"]
    .agg(lambda values: values.iloc[0] if values.nunique() == 1 else None)
)

comments_category = reviews.loc[has_message, "order_id"].map(single_category)
category_counts = comments_category.value_counts()

print(f"Categorias com comentários: {category_counts.size}")
print(f"Categorias com pelo menos 500 comentários: {(category_counts >= 500).sum()}")
category_counts.head(15)

Categorias com comentários: 73
Categorias com pelo menos 500 comentários: 19


order_id
cama_mesa_banho           4191
beleza_saude              3324
esporte_lazer             2927
informatica_acessorios    2701
moveis_decoracao          2486
relogios_presentes        2468
utilidades_domesticas     2306
telefonia                 1834
automotivo                1665
ferramentas_jardim        1531
cool_stuff                1485
brinquedos                1420
perfumaria                1224
eletronicos               1038
bebes                     1017
Name: count, dtype: int64

## 11. Comentários e notas ao longo do tempo

Acompanho o volume de comentários e a nota média por trimestre para ver se o período muda alguma coisa. Se a nota quase não variar, um filtro de período acrescenta pouco.

In [18]:
over_time = pd.DataFrame({
    "quarter": pd.to_datetime(reviews.loc[has_message, "review_creation_date"]).dt.to_period("Q"),
    "score": reviews.loc[has_message, "review_score"],
})
over_time.groupby("quarter")["score"].agg(["count", "mean"]).round(2)

,count,mean
quarter,,
2016Q4,238,3.25
2017Q1,1809,3.82
2017Q2,3858,3.75
2017Q3,5104,3.89
2017Q4,7043,3.62
2018Q1,8515,3.42
2018Q2,8707,3.66
2018Q3,5703,3.84


## 12. Palavras mais frequentes nas notas baixas e nas notas altas

Esta contagem dá uma prévia dos temas que a busca vai encontrar. Para ela fazer sentido, ignoro palavras muito comuns, como "de", "o" e "que". Essa remoção vale só aqui: o texto que vai para o embedding continua inteiro, porque o modelo precisa da frase completa para entender o sentido.

In [19]:
import re
from collections import Counter

# Palavras comuns da língua portuguesa, ignoradas apenas nesta contagem.
# "nao" fica fora da lista porque é importante para entender as reclamações.
STOPWORDS = set('''
a o as os um uma uns umas de da do das dos em na no nas nos por para pra com sem
e ou mas que se ja foi ser era sao esta estava estou tem tinha ter muito muita mais
me eu meu minha meus minhas ele ela eles elas isso esse essa este esta aquele
ao aos como quando ate tambem so bem pois porque pelo pela pelos pelas sobre
lhe nos voce voces seu sua seus suas qual quais fiz fez vou vai ainda entao
'''.split())


def top_words(texts: pd.Series, n: int = 20) -> pd.Series:
    words = Counter()
    for text in texts:
        tokens = re.findall(r"[a-z]+", normalize_for_comparison(text))
        words.update(token for token in tokens if token not in STOPWORDS and len(token) > 2)
    return pd.Series(dict(words.most_common(n)))


scores_with_message = reviews.loc[has_message, "review_score"]

pd.DataFrame({
    "Notas 1 e 2": top_words(messages[scores_with_message <= 2]).index,
    "Notas 4 e 5": top_words(messages[scores_with_message >= 4]).index,
})

,Notas 1 e 2,Notas 4 e 5
0,nao,produto
1,produto,prazo
2,recebi,antes
3,comprei,entrega
4,veio,chegou
5,entrega,bom
6,entregue,recomendo
7,chegou,otimo
8,prazo,entregue
9,compra,qualidade


## 13. Quantos comentários falam de entrega, defeito e atendimento

Faço uma estimativa por palavras-chave, que é só aproximada: um comentário pode citar "entrega" tanto para elogiar quanto para reclamar, e pode falar do tema sem usar nenhuma das palavras da lista. Uso esses números como referência para comparar com o que o RAG encontrar e para mostrar no README que o RAG descreve padrões, mas não faz contagens.

In [20]:
THEMES = {
    "Entrega": r"entreg|prazo|chegou|chegar|receb|atras|correio|transportadora",
    "Defeito ou dano": r"defeit|quebrad|danific|estragad|amassad|nao funciona",
    "Atendimento": r"atendimento|contato|resposta|respond|sac\b|suporte",
}

normalized_messages = messages.map(normalize_for_comparison)

pd.DataFrame({
    "Comentários": {theme: normalized_messages.str.contains(pattern).sum() for theme, pattern in THEMES.items()},
    "% do total": {theme: round(normalized_messages.str.contains(pattern).mean() * 100, 1) for theme, pattern in THEMES.items()},
})

,Comentários,% do total
Entrega,21893,53.4
Defeito ou dano,1046,2.6
Atendimento,1967,4.8


## 14. Tamanho do comentário por nota

Quero saber se clientes insatisfeitos escrevem textos mais longos, o que também ajuda a entender o papel dos comentários muito curtos na base.

In [21]:
lengths.groupby(scores_with_message).agg(["count", "median", "mean"]).round(1)

,count,median,mean
review_score,,,
1,8745,88.0,99.5
2,2145,87.0,96.4
3,3557,70.0,82.9
4,5976,46.0,61.4
5,20554,42.0,51.5


## 15. Amostra de comentários de cada nota

Para conhecer o conteúdo de verdade, sorteio três comentários de cada nota, deixando de fora os que não têm nenhuma letra. Uso uma semente fixa (`random_state`) para que o sorteio mostre sempre os mesmos comentários.

In [22]:
pd.set_option("display.max_colwidth", 200)

sample = (
    pd.DataFrame({"score": scores_with_message, "message": messages})[~no_letters]
    .groupby("score", group_keys=False)
    .sample(3, random_state=42)
)
sample

,score,message
13280,1,Comprei 2 cartuchos e entregaram apenas 1 deles. Esta unidade está disponível para ser devolvida. Preciso do reembolso do valor correspondente
16609,1,"Tive que ir no correio buscar a mercadoria, além de ter pago o frete para receber em minha residência."
79014,1,"Recebi e estoquei, pois pois tinha um já aberto. Acontece que comecei a usá-lo e o mesmo fica vazando. A fragrância é idêntica ao que já possuía porém não a fixação é péssima, produto suspeito."
78557,2,"Recebi o produto dia 01/12/2017, 6ª feira, porém, ao abrir verifiquei que uma das peças estava bem amassada e a outra possuía o acabamento bem ruim na extremidade, não sendo possível sua utilização."
32066,2,Comprei a escova mas nao veio a mesma q comprei pelo site.
68517,2,"O produto chegou estourado, vazando creme pelas beiradas"
46559,3,Muito mal embalado
62844,3,PROMETEU ENTREGAR ATÉ 24 AGO E HOJE 26 DE AGO NÃO RECEBI.
5038,3,Houve atraso e tive que ir três vezes no correio para poder retirar o produto que comprei. Não comprei mais quando a entrega for feita pela targaryen.
95608,4,o led do produto não acende


## 16. Comentários sobre aplicativo

Escolhi a pergunta "O que os clientes acham do aplicativo de celular da loja?" para testar o caso em que a base não tem evidência. Antes de usá-la, verifico se a base realmente quase não fala desse tema.

In [23]:
app_mentions = normalized_messages.str.contains(r"\baplicativo\b|\bapp\b")
print(f"Comentários que citam aplicativo: {app_mentions.sum()}")
messages[app_mentions].head(10)

Comentários que citam aplicativo: 41


6296                 As ferramentas do aplicativo não são eficases. Solicitei o concelamento de uma compra antes da nota fiscal ser emitida a solicitacao foi aceita porém não foi efetivada. O produto chegou .
7025          Fiz a compra em 02/08 e aguardei os 16 dias úteis para entrega previstos, porem, depois de 17 dias úteis, no aplicativo do stark que o produto está indisponível e que tenho que cancelar a compra
7173     O produto que foi entregue não foi o mesmo comprado, comprei o Lego movie do PES e chegou na minha casa o Lego Batman do PES, o aplicativo não atualizava o passo a passo do produto depois de pago. :(
7788                                                                                                                         Facilidade para uso do app, facilidade de pagamento, produto chegou antes do prazo.
7941         O que pecou que não foi culpa da stark é que os correios alegou que não havia ninguém em casa em 3 tentativas ao qual minha mãe e outros sempre em casa

## 17. Caracteres especiais

Procuro emojis, símbolos e sinais de problema de codificação, como "Ã©" no lugar de "é". O resultado mostra o que a limpeza dos textos precisa tratar.

In [24]:
import unicodedata

all_chars = Counter("".join(messages))
special = {
    char: count for char, count in all_chars.items()
    if not (char.isalnum() or char.isspace() or char in ".,;:!?()-'\"/%$@&*+=#")
}

print(f"Caracteres diferentes fora de letras, números e pontuação comum: {len(special)}")
pd.Series(special).sort_values(ascending=False).head(20).rename(
    lambda char: f"{char!r} ({unicodedata.name(char, 'sem nome')})"
)

Caracteres diferentes fora de letras, números e pontuação comum: 83


'👏' (CLAPPING HANDS SIGN)                                     221
'👍' (THUMBS UP SIGN)                                           82
'😍' (SMILING FACE WITH HEART-SHAPED EYES)                      70
'🏻' (EMOJI MODIFIER FITZPATRICK TYPE-1-2)                      35
'´' (ACUTE ACCENT)                                             20
'😉' (WINKING FACE)                                             19
'°' (DEGREE SIGN)                                              19
'😆' (SMILING FACE WITH OPEN MOUTH AND TIGHTLY-CLOSED EYES)     19
'😡' (POUTING FACE)                                             15
'🏽' (EMOJI MODIFIER FITZPATRICK TYPE-4)                        13
'👎' (THUMBS DOWN SIGN)                                         13
'🏼' (EMOJI MODIFIER FITZPATRICK TYPE-3)                        12
'😁' (GRINNING FACE WITH SMILING EYES)                          12
'😊' (SMILING FACE WITH SMILING EYES)                           12
'💓' (BEATING HEART)                                            11
'😘' (FACE 

In [25]:
has_emoji = messages.map(lambda text: any(unicodedata.category(char) == "So" for char in text))
# "Ã" sozinho não indica problema, porque aparece em palavras em maiúsculas como "NÃO".
# O sinal de erro é "Ã" seguido de outro caractere acentuado (como em "Ã©") ou a sequência "â€".
has_encoding_issue = messages.str.contains(r"Ã[\x80-\xBF©¡ª§£³µº¢]|â€")

pd.Series({
    "Comentários com emoji ou símbolo gráfico": has_emoji.sum(),
    "Comentários com sinal de problema de codificação": has_encoding_issue.sum(),
})

Comentários com emoji ou símbolo gráfico            289
Comentários com sinal de problema de codificação      0
dtype: int64

## 18. Funil da base: do total de avaliações aos documentos finais

Aplico em sequência as decisões de seleção que tomei a partir desta exploração: fico só com as avaliações que têm mensagem, descarto os comentários sem nenhuma letra e mantenho um documento por `review_id`. O último número é o tamanho da base de conhecimento que vou indexar.

In [26]:
with_message = reviews.loc[has_message].copy()
with_message["message"] = messages

with_letters = with_message[~no_letters]
final_docs = with_letters.drop_duplicates("review_id")

funnel = pd.Series({
    "Avaliações na base original": len(reviews),
    "Com mensagem escrita": len(with_message),
    "Sem os comentários sem nenhuma letra": len(with_letters),
    "Documentos finais (um por review_id)": len(final_docs),
})
pd.DataFrame({"Quantidade": funnel, "% da base original": (funnel / len(reviews) * 100).round(1)})

,Quantidade,% da base original
Avaliações na base original,99224,100.0
Com mensagem escrita,40977,41.3
Sem os comentários sem nenhuma letra,40791,41.1
Documentos finais (um por review_id),40482,40.8


## 19. Metadados disponíveis nos documentos finais

Os filtros só funcionam bem se boa parte dos documentos tiver o metadado preenchido, então calculo a categoria e a situação de entrega de cada documento final. Quando todos os pedidos ligados a uma avaliação são da mesma categoria, o documento recebe essa categoria; se há mais de uma, fica como "mista", e se não há nenhuma, como "desconhecida". A entrega segue a mesma lógica: se todos os pedidos têm a mesma situação, o documento a recebe, e se as situações divergem ou falta a data de entrega, fica como "desconhecido".

In [27]:
orders_per_review = with_letters.groupby("review_id")["order_id"].agg(list)

categories_by_order = (
    items_with_category.groupby("order_id")["product_category_name"]
    .agg(lambda values: set(values.dropna()))
)
status_by_order = delivery.set_index("order_id")["delivery_status"]


def category_for_review(order_ids: list[str]) -> str:
    found = set()
    for order_id in order_ids:
        found |= categories_by_order.get(order_id, set())
    if not found:
        return "desconhecida"
    if len(found) > 1:
        return "mista"
    return found.pop()


def delivery_for_review(order_ids: list[str]) -> str:
    statuses = {status_by_order.get(order_id, "desconhecido") for order_id in order_ids}
    return statuses.pop() if len(statuses) == 1 else "desconhecido"


final_category = orders_per_review.map(category_for_review)
final_delivery = orders_per_review.map(delivery_for_review)

category_summary = pd.Series({
    "Categoria definida": (~final_category.isin(["mista", "desconhecida"])).sum(),
    "Mista": (final_category == "mista").sum(),
    "Desconhecida": (final_category == "desconhecida").sum(),
})
pd.DataFrame({"Documentos": category_summary, "%": (category_summary / len(final_category) * 100).round(1)})

,Documentos,%
Categoria definida,38889,96.1
Mista,528,1.3
Desconhecida,1065,2.6


In [28]:
delivery_summary = final_delivery.value_counts().rename_axis("Entrega")
pd.DataFrame({"Documentos": delivery_summary, "%": (delivery_summary / len(final_delivery) * 100).round(1)})

,Documentos,%
Entrega,,
no prazo,34365,84.9
atrasado,4209,10.4
desconhecido,1908,4.7


In [29]:
score_summary = final_docs["review_score"].value_counts().sort_index().rename_axis("Nota")
pd.DataFrame({"Documentos": score_summary, "%": (score_summary / len(final_docs) * 100).round(1)})

,Documentos,%
Nota,,
1,8632,21.3
2,2115,5.2
3,3518,8.7
4,5891,14.6
5,20326,50.2


## 20. Estimativa de custo da indexação

O Titan Text Embeddings V2 cobra pelo número de tokens enviados. Token é o pedaço de texto que o modelo processa, em geral uma palavra curta ou parte de uma palavra. Como não tenho acesso ao contador de tokens do Titan, uso uma aproximação comum para textos em português, de cerca de 4 caracteres por token.

O texto de cada documento é o título somado à mensagem. Antes de citar o valor no README, vou conferir o preço por mil tokens na página de preços do Amazon Bedrock.

In [30]:
titles = final_docs["review_comment_title"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
document_texts = (titles + " " + final_docs["message"]).str.strip()

CHARS_PER_TOKEN = 4
PRICE_PER_1000_TOKENS_USD = 0.00002  # Titan Text Embeddings V2, sob demanda; conferir em aws.amazon.com/bedrock/pricing

total_chars = document_texts.str.len().sum()
estimated_tokens = total_chars / CHARS_PER_TOKEN
estimated_cost = estimated_tokens / 1000 * PRICE_PER_1000_TOKENS_USD

print(f"Documentos: {len(document_texts):,}")
print(f"Caracteres no total: {total_chars:,}")
print(f"Tokens estimados: {estimated_tokens:,.0f}")
print(f"Custo estimado de uma indexação completa: US$ {estimated_cost:.4f}")

Documentos: 40,482
Caracteres no total: 2,900,087
Tokens estimados: 725,022
Custo estimado de uma indexação completa: US$ 0.0145


## 21. Como são os títulos

Decidi juntar o título à mensagem no texto de cada documento. Aqui confiro o que os títulos costumam trazer: se acrescentam alguma informação ou se só repetem uma avaliação genérica.

In [31]:
# Títulos só com espaços contam como ausentes
with_title = final_docs[titles != ""]
print(f"Documentos finais com título: {len(with_title)} de {len(final_docs)}")

titles.loc[with_title.index].value_counts().head(15)

Documentos finais com título: 9741 de 40482


review_comment_title
Recomendo          400
Ótimo              336
Excelente          299
recomendo          278
Bom                266
Muito bom          256
super recomendo    220
Super recomendo    193
Otimo              136
10                 111
otimo               82
bom                 80
Perfeito            67
Boa                 64
Ótimo produto       63
Name: count, dtype: int64

In [32]:
with_title.assign(title=titles.loc[with_title.index])[["review_score", "title", "message"]].sample(10, random_state=42)

,review_score,title,message
32771,4,"Chinesa Boa,Já é possível","Somente uma SURPRESA: o produto não é americano, mas, CHINÊS. Pela qualidade e semelhança não é FALSO. Porém também não é autentico. Vou usa-lo, espero que não parta o encaixe plástico prá não tra..."
36322,1,produto ruim.,Produto ruim pelo preço cobrado.
75544,1,Me enganaram...,"Comprei o tapete com a metragem 2,50 x2,50 e me mandaram a metragem 2,0 x 2,50, sendo que no siteo valor dessa metragem que me enviaram está bem abaixo do valor que paguei..."
52358,3,Produto Ok,O produto é compatível com o preço.
97930,1,REPROVACAO DO PRODUTO,"A entrega foi feita corretamente, dentro do prazo. Mas a qualidade do produto nao eh das melhores.."
16012,1,não recomendo,"Meu pedido foi uma bola de futebol de campo conforme está especificado acima, só que me entregaram uma bola de futsal, eu não jogo futsal, achei que havia pedido errado, mais o erro foi da loja."
86451,5,Conforme anúncio.,Muito bom
28724,5,Ótimo,Parabéns
16460,1,"Recebi, está incompleto",Recebi apenas uma caixa de lente das três que comprei e paguei. E não recebo retorno para resolver a questão.
16236,5,Mto lindo,"Boa qualidade,tdo mto bem feito embalado Delicado e fora q o preço super compensa Para a quantidade de coisas e material .."


## Conclusão

A exploração mostrou que a base de conhecimento é bem menor do que o total de avaliações sugere. Das 99.224 avaliações, 40.977 têm mensagem escrita. Depois de descartar os comentários sem conteúdo e de manter um único documento por avaliação, chego a 40.482 documentos, cerca de 41% da base original. É esse conjunto que será indexado no Chroma.

Esses documentos não representam a satisfação média da Olist. A nota 1 corresponde a 11,5% de todas as avaliações, mas a 21,3% das que têm comentário, e os comentários de nota 1 são duas vezes mais longos que os de nota 5 (mediana de 88 contra 42 caracteres). Quem está insatisfeito escreve mais e com mais detalhe, então a base é especialmente rica para perguntas sobre problemas e reclamações.

A entrega é o tema dominante. Ela aparece no topo das palavras mais frequentes tanto nas notas baixas ("não", "recebi", "veio", "prazo") quanto nas altas ("prazo", "antes", "chegou"), e a busca por palavras-chave indica que pouco mais da metade dos comentários fala do assunto. O impacto do atraso na nota é grande: pedidos entregues no prazo têm nota média 4,29, contra 2,57 dos atrasados, e quase metade dos pedidos atrasados recebeu nota 1. Esse resultado justifica o filtro de entrega atrasada, que vai alcançar 4.209 documentos.

Os textos são curtos (metade tem até 53 caracteres e o maior tem 208), o que confirma a decisão de não dividir os comentários em pedaços. A repetição de textos, por outro lado, me fez rever uma decisão. Eu havia planejado agrupar comentários idênticos, mas um mesmo texto como "Muito bom" aparece em 305 avaliações, de 39 categorias e com notas de 3 a 5. Agrupar faria os filtros e as evidências perderem precisão, então cada avaliação continua sendo um documento, e a variedade dos resultados fica a cargo do MMR na etapa de busca.

A exploração também me levou a duas decisões novas. Passei a descartar os 186 comentários sem nenhuma letra, como "." e "...", e a usar um documento por `review_id`. Descobri que 299 avaliações aparecem ligadas a mais de um pedido, sempre com o mesmo texto, e manter as duas linhas faria a mesma avaliação aparecer duas vezes como evidência.

Os metadados estão bem preenchidos para os filtros: 96,1% dos documentos têm uma categoria definida, e só 1,3% são de pedidos com mais de uma categoria, o que confirma que excluí-los do filtro de categoria tem pouco efeito. A categoria `cama_mesa_banho`, com 4.191 comentários, tem volume de sobra para a pergunta de teste com filtro. Já a nota média varia pouco entre os trimestres (de 3,4 a 3,9), então o filtro de período deve trazer menos ganho que os de nota, categoria e entrega.

O custo de gerar os embeddings é irrelevante para o orçamento do projeto: cerca de 725 mil tokens, o que dá menos de dois centavos de dólar por indexação completa, considerando o preço do Titan Text Embeddings V2 que ainda vou conferir na página da AWS. Isso reforça a escolha de gerar os embeddings pelo Bedrock em vez de rodar um modelo na minha máquina.

Dois achados não mudam decisões, mas precisam aparecer na documentação. Os nomes das lojas parceiras foram anonimizados pela Olist com nomes de casas de Game of Thrones ("lannister", "stark", "targaryen"), e esses nomes vão surgir nas evidências. E os títulos, presentes em 9.741 documentos, são em sua maioria avaliações genéricas ("Recomendo", "Ótimo"), mas às vezes resumem o problema ("Produto fora de embalagem", "Péssimo"), o que mantém a decisão de juntá-los à mensagem.

Três pontos ficam para as próximas etapas. Os 3.671 comentários com até 10 caracteres continuam na base por enquanto, e vou verificar no teste da busca se eles ocupam vagas demais nos resultados. Também preciso decidir, na preparação dos dados, se a limpeza mantém os emojis, que aparecem em 289 comentários e às vezes carregam sentimento, como 😡 e 👎; problemas de codificação, por outro lado, não encontrei. Por último, 41 comentários citam aplicativo, e alguns falam de fato dele, então vou reavaliar, quando montar as perguntas de teste, se a pergunta sobre o aplicativo serve para o caso sem evidência ou se é melhor trocar de tema.